# Complemento de entrenamiento E2 — ablación, sanidad y búsqueda de hiperparámetros

Lo que el benchmark no cubría y exige volver a entrenar:

| bloque | qué hace | control de la guía |
|---|---|---|
| sobreajuste | las 10 redes sobreajustan un lote fijo de 4 imágenes | 11.1, prueba de sanidad (checklist 41) |
| ablación | TransUNet sin Transformer, sin conexiones de salto y sin término Dice, 5 semillas | 4.6 (checklist 61) |
| permutación | máscaras de train barajadas entre imágenes; pipeline completo | Tabla A (checklist 41 y 56) |
| hpo | búsqueda aleatoria con la misma secuencia de intentos para las 10 | 11.2 (checklist 37) |

**Todo es exploratorio**: el test del benchmark ya fue observado. Aquí no se carga el test; todo se
elige y se mide en validación. El cuaderno fue generado por `scripts/crear_colab_complemento_entreno.py`
y **no ha sido ejecutado en Colab** por quien lo preparó: los tiempos son estimados hasta que corra el piloto.

Orden: celdas 1 a 6 siempre; después el piloto; después los bloques en el orden de prioridad
(sobreajuste, ablación, permutación FCN, HPO, permutación TransUNet). Cada corrida se reanuda sola
desde su última época guardada en Drive: si la sesión se corta, vuelve a ejecutar las celdas 1 a 6 y
la del bloque.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, os, json, hashlib, zipfile, base64, datetime
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['mountpoint', '-q', '/content/drive'], check=True)
OUTPUT = Path('/content/drive/MyDrive/tesis_imo_e2_complemento_entreno_v1')
BASE_RUNS = Path('/content/drive/MyDrive/tesis_imo_e2_arquitecturas_reales_v2')  # benchmark original
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT/'logs').mkdir(exist_ok=True)
print('Salida:', OUTPUT, '| benchmark original presente:', BASE_RUNS.exists())

In [ ]:
# Mismo paquete que el benchmark v2: datos train/val, código y fuentes de terceros.
EXPECTED_ZIP_SHA256 = '954f4fe5663d9fedd676dcee4f1f7fb9086a2a7f5c1907ce902b5df4c109cb49'
ZIP = next((p for p in [Path('/content/paquete_colab_e2_v2.zip'), Path('/content/drive/MyDrive/paquete_colab_e2_v2.zip')] if p.exists()), None)
if ZIP is None: raise FileNotFoundError('Sube paquete_colab_e2_v2.zip al panel de archivos o a la raíz de Drive.')
def sha_file(p):
    h = hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda: f.read(1024*1024), b''): h.update(b)
    return h.hexdigest()
assert sha_file(ZIP) == EXPECTED_ZIP_SHA256, 'ZIP distinto del v2 del benchmark'
BASE = Path('/content/imo_e2_v2'); BASE.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    for n in z.namelist():
        if not (BASE/n).resolve().is_relative_to(BASE): raise ValueError('Ruta no válida en ZIP')
    z.extractall(BASE)
manifest = json.loads((BASE/'PACKAGE_MANIFEST.json').read_text())
for name, expected in manifest['files'].items():
    assert sha_file(BASE/name) == expected, name
PROJECT = BASE/'tesis_imo_para_codex'
DEPENDENCIAS = {"scripts/experimento_e2_colab.py": "e6b0cb2a06b2ec50592ebee30f9c0fc627b984d13845f5fbd4fa6b85bbd00f81", "scripts/modelos_e2.py": "5b74e36860a86921fddd53dc2072404267929c4c9c14d92441f2f7c4bdd9d4db", "scripts/dataset_ready.py": "31b9707837c82baef6f7adea7716f007c0d6b16056d8ed13912c14dc839e2d5f"}
for rel, esperado in DEPENDENCIAS.items():
    assert sha_file(PROJECT/rel) == esperado, f'{rel} no es el del protocolo cerrado'
print('Paquete íntegro:', len(manifest['files']), 'archivos; dependencias del runner verificadas')

In [ ]:
# Archivos nuevos del complemento, incrustados y verificados (generados desde el repositorio).
INCRUSTADOS = {
"scripts/complemento_entreno_e2.py": {
"sha256": "dfe815ac6137705098f0d966b88d081b786449a66499117dccf8e7bfd1aa8bf8",
"b64": "..."
},
"scripts/resumir_complemento_entreno_e2.py": {
"sha256": "7c45de75c89ec5dfea1708d2362a6d45f84b7cc562ab9be1f63a2e724f1a3269",
"b64": "IiIiUmVzdW1lIGVsIGNvbXBsZW1lbnRvIGRlIGVudHJlbmFtaWVudG86IHNvYnJlYWp1c3RlLCBhYmxhY2nDs24sIHBlcm11dGFjacOzbiB5IEhQTy4KCkxlZSBzb2xvIEpTT04gZGUgbGEgY2FycGV0YSBkZSBzYWxpZGEgKHksIHNpIHNlIGRhLCBsb3MgcmVzdWx0Lmpzb24geSBtZWpvcmVzIGNoZWNrcG9pbnRzIGRlCnZhbGlkYWNpw7NuIGRlbCBiZW5jaG1hcmsgb3JpZ2luYWwpLiBObyB0b2NhIGVsIHRlc3QuIEVzY3JpYmUgUkVTVU1FTi5qc29uIHkgUkVTVU1FTi5tZC4KVG9kbyBlcyBleHBsb3JhdG9yaW86IGVsIHRlc3QgZGVsIGJlbmNobWFyayB5YSBmdWUgb2JzZXJ2YWRvLgoiIiIKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBtYXRoCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IGRlZmF1bHRkaWN0CmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCgpQUk9KRUNUID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMV0KQ09ORklHID0gUFJPSkVDVC8nY29uZmlnL2NvbXBsZW1lbnRvX2VudHJlbm9fZTIuanNvbicKU0VNSUxMQV9SRU1VRVNUUkVPID0gMjAyNjExMjMKCgpkZWYgbGVlcihwYXRoKToKICAgIHJldHVybiBqc29uLmxvYWRzKFBhdGgocGF0aCkucmVhZF90ZXh0KGVuY29kaW5nPSd1dGYtOCcpKQoKCmRlZiByZXN1bHRhZG9zKHNhbGlkYSwgcGF0cm9uKToKICAgIG91dCA9IFtdCiAgICBmb3IgcCBpbiBzb3J0ZWQoUGF0aChzYWxpZGEpLmdsb2IocGF0cm9uKSk6CiAgICAgICAgciA9IGxlZXIocCkKICAgICAgICBpZiByLmdldCgnY29tcGxldGVkJyk6CiAgICAgICAgICAgIHJbJ19kaXInXSA9IHAucGFyZW50CiAgICAgICAgICAgIG91dC5hcHBlbmQocikKICAgIHJldHVybiBvdXQKCgpkZWYgaG9sbShwdmFsb3Jlcyk6CiAgICBvcmRlbiA9IHNvcnRlZChyYW5nZShsZW4ocHZhbG9yZXMpKSwga2V5PWxhbWJkYSBpOiBwdmFsb3Jlc1tpXSkKICAgIGFqdXN0ID0gW05vbmVdKmxlbihwdmFsb3Jlcyk7IHByZXZpbyA9IDAuCiAgICBmb3IgcmFuZ28sIGkgaW4gZW51bWVyYXRlKG9yZGVuKToKICAgICAgICBwcmV2aW8gPSBtYXgocHJldmlvLCBtaW4oMS4sIChsZW4ocHZhbG9yZXMpLXJhbmdvKSpwdmFsb3Jlc1tpXSkpCiAgICAgICAgYWp1c3RbaV0gPSBwcmV2aW8KICAgIHJldHVybiBhanVzdAoKCmRlZiBwb3JfZ3J1cG8oZGV0YWxsZSk6CiAgICBnID0gZGVmYXVsdGRpY3QobGlzdCkKICAgIGZvciBkIGluIGRldGFsbGU6CiAgICAgICAgZ1tkWydncm91cF9pZCddXS5hcHBlbmQoZFsnRGljZSddKQogICAgcmV0dXJuIHtrOiBmbG9hdChucC5tZWFuKHYpKSBmb3IgaywgdiBpbiBnLml0ZW1zKCl9CgoKZGVmIHZhbGlkYWNpb25fb3JpZ2luYWwoYmFzZV9ydW5zLCBzZW1pbGxhKToKICAgICIiIkRldGFsbGUgcG9yIGltYWdlbiBkZSB2YWxpZGFjacOzbiBkZWwgbWVqb3IgY2hlY2twb2ludCBvcmlnaW5hbCAoc29sbyBzaSBleGlzdGUpLiIiIgogICAgaWYgbm90IGJhc2VfcnVuczoKICAgICAgICByZXR1cm4gTm9uZQogICAgaW1wb3J0IHRvcmNoCiAgICBjYXJwZXRhID0gUGF0aChiYXNlX3J1bnMpL2YndHJhbnN1bmV0XzJkX3NlZWR7c2VtaWxsYX0nCiAgICByZXMgPSBjYXJwZXRhLydyZXN1bHQuanNvbicKICAgIGlmIG5vdCByZXMuZXhpc3RzKCk6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIHIgPSBsZWVyKHJlcykKICAgIHRyeToKICAgICAgICBja3B0ID0gdG9yY2gubG9hZChjYXJwZXRhL3JbJ2Jlc3RfZmlsZSddLCBtYXBfbG9jYXRpb249J2NwdScsIHdlaWdodHNfb25seT1UcnVlKQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlcnJvcjogICMgbm9xYTogQkxFMDAxIC0gc2UgcmVwb3J0YSB5IHNlIHNpZ3VlIHNpbiBlbCBib290c3RyYXAgcG9yIGdydXBvcwogICAgICAgIHByaW50KGYnTm8gc2UgcHVkbyBsZWVyIGxhIHZhbGlkYWNpw7NuIG9yaWdpbmFsIGRlIGxhIHNlbWlsbGEge3NlbWlsbGF9OiB7ZXJyb3J9JykKICAgICAgICByZXR1cm4gTm9uZQogICAgcmV0dXJuIHsnRGljZSc6IHJbJ2Jlc3RfdmFsX2dyb3VwX0RpY2UnXSwgJ3ZhbGlkYXRpb24nOiBja3B0Wyd2YWxpZGF0aW9uJ119CgoKZGVmIGluZmVyZW5jaWFfcGFyZWFkYShkaWZfc2VtaWxsYSwgZGlmX2dydXBvLCBybmcpOgogICAgZnJvbSBzY2lweSBpbXBvcnQgc3RhdHMKICAgIGQgPSBucC5hc2FycmF5KGRpZl9zZW1pbGxhLCBkdHlwZT1mbG9hdCk7IG4gPSBsZW4oZCkKICAgIG91dCA9IHsnbl9zZW1pbGxhcyc6IG4sICdkaWZlcmVuY2lhX21lZGlhJzogZmxvYXQoZC5tZWFuKCkpIGlmIG4gZWxzZSBOb25lfQogICAgaWYgbiA+PSAyOgogICAgICAgIHNkID0gZmxvYXQoZC5zdGQoZGRvZj0xKSk7IHNlID0gc2QvbWF0aC5zcXJ0KG4pOyB0Y3JpdCA9IHN0YXRzLnQucHBmKC45NzUsIG4tMSkKICAgICAgICBvdXQudXBkYXRlKHsnZGUnOiBzZCwgJ2ljOTVfdCc6IFtmbG9hdChkLm1lYW4oKS10Y3JpdCpzZSksIGZsb2F0KGQubWVhbigpK3Rjcml0KnNlKV0sCiAgICAgICAgICAgICAgICAgICAgJ3BfdF9wYXJlYWRhJzogZmxvYXQoc3RhdHMudHRlc3RfMXNhbXAoZCwgMC4pLnB2YWx1ZSkgaWYgc2QgPiAwIGVsc2UgTm9uZX0pCiAgICAgICAgaWYgbiA+PSA1IGFuZCBucC5hbnkoZCAhPSAwKToKICAgICAgICAgICAgb3V0WydwX3dpbGNveG9uX2V4YWN0byddID0gZmxvYXQoc3RhdHMud2lsY294b24oZCwgbWV0aG9kPSdleGFjdCcpLnB2YWx1ZSkKICAgICAgICAgICAgb3V0Wydub3RhX3dpbGNveG9uJ10gPSBmJ0NvbiB7bn0gcGFyZXMgZWwgcCBtw61uaW1vIGJpbGF0ZXJhbCBlcyB7Mi8yKipuOi40Zn0nCiAgICBpZiBkaWZfZ3J1cG8gaXMgbm90IE5vbmU6CiAgICAgICAgZyA9IG5wLmFzYXJyYXkoZGlmX2dydXBvLCBkdHlwZT1mbG9hdCkKICAgICAgICBib290cyA9IG5wLmFycmF5KFtnW3JuZy5pbnRlZ2VycygwLCBsZW4oZyksIGxlbihnKSldLm1lYW4oKSBmb3IgXyBpbiByYW5nZSgyMDAwKV0pCiAgICAgICAgc2lnbm9zID0gcm5nLmNob2ljZShbLTEuLCAxLl0sIHNpemU9KDEwMDAwLCBsZW4oZykpKQogICAgICAgIG51bGFzID0gbnAuYWJzKChzaWdub3MqZykubWVhbihheGlzPTEpKQogICAgICAgIG91dC51cGRhdGUoeyduX2dydXBvcyc6IGxlbihnKSwgJ2RpZmVyZW5jaWFfbWVkaWFfZ3J1cG9zJzogZmxvYXQoZy5tZWFuKCkpLAogICAgICAgICAgICAgICAgICAgICdpYzk1X2Jvb3RzdHJhcF9ncnVwb3MnOiBbZmxvYXQobnAucXVhbnRpbGUoYm9vdHMsIC4wMjUpKSwgZmxvYXQobnAucXVhbnRpbGUoYm9vdHMsIC45NzUpKV0sCiAgICAgICAgICAgICAgICAgICAgJ3Bfc2lnbm9zX2dydXBvcyc6IGZsb2F0KCgxK25wLnN1bShudWxhcyA+PSBhYnMoZy5tZWFuKCkpKSkvKDEwMDAwKzEpKX0pCiAgICByZXR1cm4gb3V0CgoKZGVmIHJlc3VtaXJfYWJsYWNpb24oc2FsaWRhLCBjZmcsIGJhc2VfcnVucywgcm5nKToKICAgIHNlbWlsbGFzID0gY2ZnWydjb211biddWydzZW1pbGxhcyddCiAgICBmaWxhcyA9IHsoclsndmFyaWFudGUnXSwgclsnc2VtaWxsYSddKTogciBmb3IgciBpbiByZXN1bHRhZG9zKHNhbGlkYSwgJ2FibF8qL3Jlc3VsdC5qc29uJyl9CiAgICByZWZlcmVuY2lhX2Z1ZW50ZSA9ICdjb250cm9sIGNvbXBsZXRvIGVuIGVzdGEgbWlzbWEgY2FycGV0YScgaWYgYWxsKAogICAgICAgICgnY29tcGxldG8nLCBzKSBpbiBmaWxhcyBmb3IgcyBpbiBzZW1pbGxhcykgZWxzZSAnYmVuY2htYXJrIG9yaWdpbmFsJwogICAgcmVmX2RpY2UsIHJlZl9ncnVwbyA9IHt9LCB7fQogICAgZm9yIHMgaW4gc2VtaWxsYXM6CiAgICAgICAgaWYgcmVmZXJlbmNpYV9mdWVudGUuc3RhcnRzd2l0aCgnY29udHJvbCcpOgogICAgICAgICAgICByID0gZmlsYXNbKCdjb21wbGV0bycsIHMpXQogICAgICAgICAgICByZWZfZGljZVtzXSA9IHJbJ2Jlc3RfdmFsX2dyb3VwX0RpY2UnXQogICAgICAgICAgICByZWZfZ3J1cG9bc10gPSBwb3JfZ3J1cG8obGVlcihyWydfZGlyJ10vJ2Jlc3RfdmFsX3ByZWRpY3Rpb25zLmpzb24nKVsndmFsaWRhdGlvbiddKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIG9yaWcgPSB2YWxpZGFjaW9uX29yaWdpbmFsKGJhc2VfcnVucywgcykKICAgICAgICAgICAgcmVmX2RpY2Vbc10gPSBvcmlnWydEaWNlJ10gaWYgb3JpZyBlbHNlIGNmZ1sncmVmZXJlbmNpYV9vcmlnaW5hbCddWydkaWNlX3ZhbGlkYWNpb25fcG9yX3NlbWlsbGEnXVsndHJhbnN1bmV0XzJkJ11bc3RyKHMpXQogICAgICAgICAgICBpZiBvcmlnOgogICAgICAgICAgICAgICAgcmVmX2dydXBvW3NdID0gcG9yX2dydXBvKG9yaWdbJ3ZhbGlkYXRpb24nXSkKICAgIHZhcmlhbnRlcyA9IHt9CiAgICBmb3IgdmFyIGluIGNmZ1snYWJsYWNpb24nXVsndmFyaWFudGVzJ106CiAgICAgICAgcGFyZXMgPSBbKHMsIGZpbGFzWyh2YXIsIHMpXSkgZm9yIHMgaW4gc2VtaWxsYXMgaWYgKHZhciwgcykgaW4gZmlsYXNdCiAgICAgICAgaWYgbm90IHBhcmVzOgogICAgICAgICAgICB2YXJpYW50ZXNbdmFyXSA9IHsnY29ycmlkYXMnOiAwfQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGRpZiA9IFtyWydiZXN0X3ZhbF9ncm91cF9EaWNlJ10tcmVmX2RpY2Vbc10gZm9yIHMsIHIgaW4gcGFyZXNdCiAgICAgICAgZGlmX2dydXBvID0gTm9uZQogICAgICAgIGlmIGFsbChzIGluIHJlZl9ncnVwbyBmb3IgcywgXyBpbiBwYXJlcyk6CiAgICAgICAgICAgIHZhcl9nID0gW3Bvcl9ncnVwbyhsZWVyKHJbJ19kaXInXS8nYmVzdF92YWxfcHJlZGljdGlvbnMuanNvbicpWyd2YWxpZGF0aW9uJ10pIGZvciBfLCByIGluIHBhcmVzXQogICAgICAgICAgICBncnVwb3MgPSBzb3J0ZWQoc2V0LmludGVyc2VjdGlvbigqW3NldCh2KSBmb3IgdiBpbiB2YXJfZ10sICpbc2V0KHJlZl9ncnVwb1tzXSkgZm9yIHMsIF8gaW4gcGFyZXNdKSkKICAgICAgICAgICAgZGlmX2dydXBvID0gW25wLm1lYW4oW3ZnW2ddIGZvciB2ZyBpbiB2YXJfZ10pIC0gbnAubWVhbihbcmVmX2dydXBvW3NdW2ddIGZvciBzLCBfIGluIHBhcmVzXSkKICAgICAgICAgICAgICAgICAgICAgICAgIGZvciBnIGluIGdydXBvc10KICAgICAgICB2YXJpYW50ZXNbdmFyXSA9IHsnY29ycmlkYXMnOiBsZW4ocGFyZXMpLAogICAgICAgICAgICAgICAgICAgICAgICAgICdkaWNlX3Bvcl9zZW1pbGxhJzoge3M6IHJbJ2Jlc3RfdmFsX2dyb3VwX0RpY2UnXSBmb3IgcywgciBpbiBwYXJlc30sCiAgICAgICAgICAgICAgICAgICAgICAgICAgJ2RpY2VfbWVkaW8nOiBmbG9hdChucC5tZWFuKFtyWydiZXN0X3ZhbF9ncm91cF9EaWNlJ10gZm9yIF8sIHIgaW4gcGFyZXNdKSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgJ2Vwb2Nhcyc6IHtzOiByWydlcG9jaHNfY29tcGxldGVkJ10gZm9yIHMsIHIgaW4gcGFyZXN9LAogICAgICAgICAgICAgICAgICAgICAgICAgICdwYXJhbWV0cm9zJzogcGFyZXNbMF1bMV1bJ3BhcmFtZXRlcnMnXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAnaW5mZXJlbmNpYSc6IGluZmVyZW5jaWFfcGFyZWFkYShkaWYsIGRpZl9ncnVwbywgcm5nKX0KICAgIGNvbXBsZXRhcyA9IFt2IGZvciB2IGluIHZhcmlhbnRlcy52YWx1ZXMoKSBpZiB2LmdldCgnY29ycmlkYXMnKSA9PSBsZW4oc2VtaWxsYXMpXQogICAgcHYgPSBbdlsnaW5mZXJlbmNpYSddLmdldCgncF90X3BhcmVhZGEnKSBmb3IgdiBpbiBjb21wbGV0YXNdCiAgICBpZiBjb21wbGV0YXMgYW5kIGFsbChwIGlzIG5vdCBOb25lIGZvciBwIGluIHB2KToKICAgICAgICBmb3IgdiwgcCBpbiB6aXAoY29tcGxldGFzLCBob2xtKHB2KSk6CiAgICAgICAgICAgIHZbJ2luZmVyZW5jaWEnXVsncF90X3BhcmVhZGFfaG9sbSddID0gcAogICAgcmV0dXJuIHsncmVmZXJlbmNpYSc6IHJlZmVyZW5jaWFfZnVlbnRlLCAncmVmZXJlbmNpYV9kaWNlX3Bvcl9zZW1pbGxhJzogcmVmX2RpY2UsICd2YXJpYW50ZXMnOiB2YXJpYW50ZXMsCiAgICAgICAgICAgICdsZWN0dXJhJzogJ0RpZmVyZW5jaWEgPSB2YXJpYW50ZSAtIHJlZmVyZW5jaWEgZW4gRGljZSBkZSB2YWxpZGFjacOzbi4gTmVnYXRpdmE6IHF1aXRhciBlbCBjb21wb25lbnRlIGVtcGVvcmEuJ30KCgpkZWYgcmVzdW1pcl9wZXJtdXRhY2lvbihzYWxpZGEsIGNmZywgYmFzZV9ydW5zKToKICAgIG91dCA9IHt9CiAgICBmb3IgYmxvcXVlLCBzcGVjIGluIGNmZ1sncGVybXV0YWNpb24nXVsnYmxvcXVlcyddLml0ZW1zKCk6CiAgICAgICAgbW9kZWxvID0gc3BlY1snbW9kZWxvJ10KICAgICAgICBmaWxhcyA9IHJlc3VsdGFkb3Moc2FsaWRhLCBmJ3Blcm1fe21vZGVsb31fYiovcmVzdWx0Lmpzb24nKQogICAgICAgIGlmIG5vdCBmaWxhczoKICAgICAgICAgICAgb3V0W2Jsb3F1ZV0gPSB7J0JfYWxjYW56YWRvJzogMH0KICAgICAgICAgICAgY29udGludWUKICAgICAgICBzZW1pbGxhID0gY2ZnWydwZXJtdXRhY2lvbiddWydzZW1pbGxhX2VudHJlbmFtaWVudG8nXQogICAgICAgIHNfb2JzLCBmdWVudGUgPSBOb25lLCBOb25lCiAgICAgICAgaWYgYmFzZV9ydW5zIGFuZCAoUGF0aChiYXNlX3J1bnMpL2Yne21vZGVsb31fc2VlZHtzZW1pbGxhfS9yZXN1bHQuanNvbicpLmV4aXN0cygpOgogICAgICAgICAgICBzX29icyA9IGxlZXIoUGF0aChiYXNlX3J1bnMpL2Yne21vZGVsb31fc2VlZHtzZW1pbGxhfS9yZXN1bHQuanNvbicpWydiZXN0X3ZhbF9ncm91cF9EaWNlJ10KICAgICAgICAgICAgZnVlbnRlID0gJ3Jlc3VsdC5qc29uIG9yaWdpbmFsIGVuIERyaXZlJwogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNfb2JzID0gY2ZnWydyZWZlcmVuY2lhX29yaWdpbmFsJ11bJ2RpY2VfdmFsaWRhY2lvbl9wb3Jfc2VtaWxsYSddW21vZGVsb11bc3RyKHNlbWlsbGEpXQogICAgICAgICAgICBmdWVudGUgPSAnYXVkaXRvcmlhX2UyX3JlY2liaWRhIChjb3BpYSBlbiBsYSBjb25maWd1cmFjacOzbiknCiAgICAgICAgc19iID0gbnAuYXJyYXkoW3JbJ2Jlc3RfdmFsX2dyb3VwX0RpY2UnXSBmb3IgciBpbiBmaWxhc10pCiAgICAgICAgQiA9IGxlbihzX2IpCiAgICAgICAgb3V0W2Jsb3F1ZV0gPSB7J21vZGVsbyc6IG1vZGVsbywgJ0JfZGlzZW5hZG8nOiBzcGVjWydCJ10sICdCX2FsY2FuemFkbyc6IEIsCiAgICAgICAgICAgICAgICAgICAgICAgJ3Nfb2JzJzogc19vYnMsICdmdWVudGVfc19vYnMnOiBmdWVudGUsCiAgICAgICAgICAgICAgICAgICAgICAgJ3AnOiBmbG9hdCgoMStucC5zdW0oc19iID49IHNfb2JzKSkvKEIrMSkpLCAncF9taW5pbW9fcG9zaWJsZSc6IDEvKEIrMSksCiAgICAgICAgICAgICAgICAgICAgICAgJ3NfYic6IHsnbWluJzogZmxvYXQoc19iLm1pbigpKSwgJ21lZGlhJzogZmxvYXQoc19iLm1lYW4oKSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAnbWVkaWFuYSc6IGZsb2F0KG5wLm1lZGlhbihzX2IpKSwgJ21heCc6IGZsb2F0KHNfYi5tYXgoKSl9LAogICAgICAgICAgICAgICAgICAgICAgICdlcG9jYXNfbWVkaWFzJzogZmxvYXQobnAubWVhbihbclsnZXBvY2hzX2NvbXBsZXRlZCddIGZvciByIGluIGZpbGFzXSkpLAogICAgICAgICAgICAgICAgICAgICAgICdjdW1wbGVfbWluaW1vX2d1aWFfY29zdG9zbyc6IEIgPj0gMjAwfQogICAgcmV0dXJuIG91dAoKCmRlZiByZXN1bWlyX2hwbyhzYWxpZGEsIGNmZyk6CiAgICBmaWxhcyA9IHJlc3VsdGFkb3Moc2FsaWRhLCAnaHBvXyovcmVzdWx0Lmpzb24nKQogICAgbW9kZWxvcyA9IGNmZ1snc29icmVhanVzdGUnXVsnbW9kZWxvcyddCiAgICBwb3JfdCA9IGRlZmF1bHRkaWN0KGRpY3QpCiAgICBmb3IgciBpbiBmaWxhczoKICAgICAgICBwb3JfdFtyWydpbnRlbnRvJ11bJ3QnXV1bclsnbW9kZWxvJ11dID0gcgogICAgVF9jb21wbGV0byA9IDAKICAgIHdoaWxlIFRfY29tcGxldG8gaW4gcG9yX3QgYW5kIGFsbChtIGluIHBvcl90W1RfY29tcGxldG9dIGZvciBtIGluIG1vZGVsb3MpOgogICAgICAgIFRfY29tcGxldG8gKz0gMQogICAgdGFibGEgPSB7bTogW10gZm9yIG0gaW4gbW9kZWxvc30KICAgIGZvciB0IGluIHJhbmdlKFRfY29tcGxldG8pOgogICAgICAgIGZvciBtIGluIG1vZGVsb3M6CiAgICAgICAgICAgIHIgPSBwb3JfdFt0XVttXQogICAgICAgICAgICB0YWJsYVttXS5hcHBlbmQoeyd0JzogdCwgJ2xyJzogclsnb3B0aW0nXVsnbHInXSwgJ3dlaWdodF9kZWNheSc6IHJbJ29wdGltJ10uZ2V0KCd3ZWlnaHRfZGVjYXknKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAnZGljZV92YWwnOiByWydiZXN0X3ZhbF9ncm91cF9EaWNlJ10sICdtZWpvcl9lcG9jYSc6IHJbJ2Jlc3RfZXBvY2gnXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAnZXBvY2FzJzogclsnZXBvY2hzX2NvbXBsZXRlZCddfSkKICAgIGdhbmFkb3JlcyA9IHt9CiAgICBmb3IgbSwgZmlsYXNfbSBpbiB0YWJsYS5pdGVtcygpOgogICAgICAgIGlmIGZpbGFzX206CiAgICAgICAgICAgIGcgPSBtYXgoZmlsYXNfbSwga2V5PWxhbWJkYSBmOiBmWydkaWNlX3ZhbCddKQogICAgICAgICAgICByZWNldGEgPSBmaWxhc19tWzBdWydkaWNlX3ZhbCddCiAgICAgICAgICAgIGdhbmFkb3Jlc1ttXSA9IHsqKmcsICdkaWNlX3JlY2V0YV9taXNtb19wcmVzdXB1ZXN0byc6IHJlY2V0YSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICdtZWpvcmFfc29icmVfcmVjZXRhJzogZ1snZGljZV92YWwnXS1yZWNldGEsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAncmFuZ29fZW50cmVfaW50ZW50b3MnOiBtYXgoZlsnZGljZV92YWwnXSBmb3IgZiBpbiBmaWxhc19tKS1taW4oZlsnZGljZV92YWwnXSBmb3IgZiBpbiBmaWxhc19tKX0KICAgIHJldHVybiB7J1RfY29tcGxldG9fcGFyYV9sYXNfMTAnOiBUX2NvbXBsZXRvLAogICAgICAgICAgICAndGFyZWFzX2NvbXBsZXRhcyc6IGxlbihmaWxhcyksICd0b2Rhc19sYXNfY29uZmlndXJhY2lvbmVzJzogdGFibGEsICdnYW5hZG9yX3Bvcl92YWxpZGFjaW9uJzogZ2FuYWRvcmVzLAogICAgICAgICAgICAnYWR2ZXJ0ZW5jaWEnOiAnVW5hIHNlbWlsbGEgeSAxMCDDqXBvY2FzIHBvciBpbnRlbnRvOiBlbCBnYW5hZG9yIGxsZXZhIHJ1aWRvIGRlIHNlbWlsbGEuIEV4cGxvcmF0b3Jpby4nfQoKCmRlZiByZXN1bWlyX3NvYnJlYWp1c3RlKHNhbGlkYSk6CiAgICBvdXQgPSB7fQogICAgZm9yIHAgaW4gc29ydGVkKChQYXRoKHNhbGlkYSkvJ3NvYnJlYWp1c3RlJykuZ2xvYignKi5qc29uJykpOgogICAgICAgIHIgPSBsZWVyKHApCiAgICAgICAgb3V0W3JbJ21vZGVsbyddXSA9IHtrOiByW2tdIGZvciBrIGluICgnZGljZV9sb3RlX21vZG9fZXZhbCcsICdwZXJkaWRhX2luaWNpYWwnLCAncGVyZGlkYV9maW5hbCcsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAncGFzYScsICd1bWJyYWwnLCAncGFzb3MnLCAnc2VndW5kb3MnKX0KICAgIHJldHVybiBvdXQKCgpkZWYgbWFya2Rvd24ocmVzKToKICAgIEwgPSBbJyMgQ29tcGxlbWVudG8gZGUgZW50cmVuYW1pZW50byBFMiDigJQgcmVzdW1lbicsICcnLAogICAgICAgICAnR2VuZXJhZG8gcG9yIGBzY3JpcHRzL3Jlc3VtaXJfY29tcGxlbWVudG9fZW50cmVub19lMi5weWAuIEV4cGxvcmF0b3JpbzogZWwgdGVzdCBkZWwgYmVuY2htYXJrIHlhICcKICAgICAgICAgJ2Z1ZSBvYnNlcnZhZG87IGFxdcOtIHRvZG8gc2UgZWxpZ2UgeSBzZSBtaWRlIGVuIHZhbGlkYWNpw7NuLicsICcnXQogICAgcyA9IHJlc1snc29icmVhanVzdGUnXQogICAgTCArPSBbJyMjIFNvYnJlYWp1c3RlIGEgdW4gbG90ZSBkZSA0IGltw6FnZW5lcycsICcnXQogICAgaWYgczoKICAgICAgICBMICs9IFsnfCByZWQgfCBEaWNlIGxvdGUgfCBww6lyZGlkYSBpbmljaWFsIHwgcMOpcmRpZGEgZmluYWwgfCBwYXNhIHwnLCAnfC0tLXwtLS18LS0tfC0tLXwtLS18J10KICAgICAgICBMICs9IFtmJ3wge219IHwge3ZbImRpY2VfbG90ZV9tb2RvX2V2YWwiXTouNGZ9IHwge3ZbInBlcmRpZGFfaW5pY2lhbCJdOi40Zn0gfCB7dlsicGVyZGlkYV9maW5hbCJdOi40Zn0gfCAnCiAgICAgICAgICAgICAgZid7InPDrSIgaWYgdlsicGFzYSJdIGVsc2UgIm5vIn0gfCcgZm9yIG0sIHYgaW4gcy5pdGVtcygpXQogICAgZWxzZToKICAgICAgICBMLmFwcGVuZCgnU2luIGNvcnJpZGFzLicpCiAgICBhID0gcmVzWydhYmxhY2lvbiddCiAgICBMICs9IFsnJywgZicjIyBBYmxhY2nDs24gZGUgVHJhbnNVTmV0IChyZWZlcmVuY2lhOiB7YVsicmVmZXJlbmNpYSJdfSknLCAnJywKICAgICAgICAgICd8IHZhcmlhbnRlIHwgY29ycmlkYXMgfCBEaWNlIG1lZGlvIHwgZGlmZXJlbmNpYSB8IElDOTUgdCB8IHAgdCB8IHAgSG9sbSB8IHAgV2lsY294b24gfCBJQzk1IGdydXBvcyB8JywKICAgICAgICAgICd8LS0tfC0tLXwtLS18LS0tfC0tLXwtLS18LS0tfC0tLXwtLS18J10KICAgIGZvciB2YXIsIHYgaW4gYVsndmFyaWFudGVzJ10uaXRlbXMoKToKICAgICAgICBpZiBub3Qgdi5nZXQoJ2NvcnJpZGFzJyk6CiAgICAgICAgICAgIEwuYXBwZW5kKGYnfCB7dmFyfSB8IDAgfCB8IHwgfCB8IHwgfCB8Jyk7IGNvbnRpbnVlCiAgICAgICAgaSA9IHZbJ2luZmVyZW5jaWEnXQoKICAgICAgICBkZWYgZih4KToKICAgICAgICAgICAgcmV0dXJuICcnIGlmIHggaXMgTm9uZSBlbHNlIGYne3g6LjRmfScKCiAgICAgICAgZGVmIGljKHgpOgogICAgICAgICAgICByZXR1cm4gJycgaWYgbm90IHggZWxzZSBmJ1t7eFswXTouNGZ9OyB7eFsxXTouNGZ9XScKICAgICAgICBMLmFwcGVuZChmJ3wge3Zhcn0gfCB7dlsiY29ycmlkYXMiXX0gfCB7dlsiZGljZV9tZWRpbyJdOi40Zn0gfCB7ZihpLmdldCgiZGlmZXJlbmNpYV9tZWRpYSIpKX0gfCAnCiAgICAgICAgICAgICAgICAgZid7aWMoaS5nZXQoImljOTVfdCIpKX0gfCB7ZihpLmdldCgicF90X3BhcmVhZGEiKSl9IHwge2YoaS5nZXQoInBfdF9wYXJlYWRhX2hvbG0iKSl9IHwgJwogICAgICAgICAgICAgICAgIGYne2YoaS5nZXQoInBfd2lsY294b25fZXhhY3RvIikpfSB8IHtpYyhpLmdldCgiaWM5NV9ib290c3RyYXBfZ3J1cG9zIikpfSB8JykKICAgIEwgKz0gWycnLCAnIyMgUGVybXV0YWNpw7NuIGRlIG3DoXNjYXJhcyBlbiB0cmFpbicsICcnXQogICAgZm9yIGIsIHYgaW4gcmVzWydwZXJtdXRhY2lvbiddLml0ZW1zKCk6CiAgICAgICAgaWYgbm90IHYuZ2V0KCdCX2FsY2FuemFkbycpOgogICAgICAgICAgICBMLmFwcGVuZChmJy0ge2J9OiBzaW4gcsOpcGxpY2FzIGNvbXBsZXRhcy4nKTsgY29udGludWUKICAgICAgICBMLmFwcGVuZChmJy0ge2J9ICh7dlsibW9kZWxvIl19KTogQiA9IHt2WyJCX2FsY2FuemFkbyJdfSBkZSB7dlsiQl9kaXNlbmFkbyJdfTsgc19vYnMgPSB7dlsic19vYnMiXTouNGZ9OyAnCiAgICAgICAgICAgICAgICAgZidzX2IgbWVkaWEge3ZbInNfYiJdWyJtZWRpYSJdOi40Zn0gKG3DoXguIHt2WyJzX2IiXVsibWF4Il06LjRmfSk7IHAgPSB7dlsicCJdOi40Zn0gJwogICAgICAgICAgICAgICAgIGYnKG3DrW5pbW8gcG9zaWJsZSB7dlsicF9taW5pbW9fcG9zaWJsZSJdOi40Zn0pLicpCiAgICBoID0gcmVzWydocG8nXQogICAgTCArPSBbJycsIGYnIyMgQsO6c3F1ZWRhIGRlIGhpcGVycGFyw6FtZXRyb3MgKFQgY29tcGxldG8gcGFyYSBsYXMgMTA6IHtoWyJUX2NvbXBsZXRvX3BhcmFfbGFzXzEwIl19KScsICcnXQogICAgaWYgaFsnZ2FuYWRvcl9wb3JfdmFsaWRhY2lvbiddOgogICAgICAgIEwgKz0gWyd8IHJlZCB8IHQgZ2FuYWRvciB8IGxyIHwgd2VpZ2h0IGRlY2F5IHwgRGljZSB2YWwgfCByZWNldGEgbWlzbW8gcHJlc3VwdWVzdG8gfCByYW5nbyB8JywKICAgICAgICAgICAgICAnfC0tLXwtLS18LS0tfC0tLXwtLS18LS0tfC0tLXwnXQogICAgICAgIEwgKz0gW2YnfCB7bX0gfCB7Z1sidCJdfSB8IHtnWyJsciJdOi4yZX0gfCB7Z1sid2VpZ2h0X2RlY2F5Il19IHwge2dbImRpY2VfdmFsIl06LjRmfSB8ICcKICAgICAgICAgICAgICBmJ3tnWyJkaWNlX3JlY2V0YV9taXNtb19wcmVzdXB1ZXN0byJdOi40Zn0gfCB7Z1sicmFuZ29fZW50cmVfaW50ZW50b3MiXTouNGZ9IHwnCiAgICAgICAgICAgICAgZm9yIG0sIGcgaW4gaFsnZ2FuYWRvcl9wb3JfdmFsaWRhY2lvbiddLml0ZW1zKCldCiAgICBMICs9IFsnJywgaFsnYWR2ZXJ0ZW5jaWEnXSwgJyddCiAgICByZXR1cm4gJ1xuJy5qb2luKEwpCgoKZGVmIG1haW4oYXJndj1Ob25lKToKICAgIHAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBwLmFkZF9hcmd1bWVudCgnLS1vdXRwdXQnLCByZXF1aXJlZD1UcnVlKQogICAgcC5hZGRfYXJndW1lbnQoJy0tYmFzZS1ydW5zJywgaGVscD0nQ2FycGV0YSBkZWwgYmVuY2htYXJrIG9yaWdpbmFsIChyZXN1bHQuanNvbiB5IG1lam9yZXMgY2hlY2twb2ludHMpJykKICAgIHAuYWRkX2FyZ3VtZW50KCctLWNvbmZpZycsIGRlZmF1bHQ9c3RyKENPTkZJRykpCiAgICBhcmdzID0gcC5wYXJzZV9hcmdzKGFyZ3YpCiAgICBjZmcgPSBsZWVyKGFyZ3MuY29uZmlnKQogICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKFNFTUlMTEFfUkVNVUVTVFJFTykKICAgIHJlcyA9IHsnZXhwbG9yYXRvcmlvJzogVHJ1ZSwgJ3Rlc3RfZXZhbHVhZG8nOiBGYWxzZSwKICAgICAgICAgICAnc29icmVhanVzdGUnOiByZXN1bWlyX3NvYnJlYWp1c3RlKGFyZ3Mub3V0cHV0KSwKICAgICAgICAgICAnYWJsYWNpb24nOiByZXN1bWlyX2FibGFjaW9uKGFyZ3Mub3V0cHV0LCBjZmcsIGFyZ3MuYmFzZV9ydW5zLCBybmcpLAogICAgICAgICAgICdwZXJtdXRhY2lvbic6IHJlc3VtaXJfcGVybXV0YWNpb24oYXJncy5vdXRwdXQsIGNmZywgYXJncy5iYXNlX3J1bnMpLAogICAgICAgICAgICdocG8nOiByZXN1bWlyX2hwbyhhcmdzLm91dHB1dCwgY2ZnKX0KICAgIHNhbGlkYSA9IFBhdGgoYXJncy5vdXRwdXQpCiAgICAoc2FsaWRhLydSRVNVTUVOLmpzb24nKS53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVzLCBpbmRlbnQ9MiwgZW5zdXJlX2FzY2lpPUZhbHNlLCBkZWZhdWx0PXN0ciksIGVuY29kaW5nPSd1dGYtOCcpCiAgICAoc2FsaWRhLydSRVNVTUVOLm1kJykud3JpdGVfdGV4dChtYXJrZG93bihyZXMpLCBlbmNvZGluZz0ndXRmLTgnKQogICAgcHJpbnQobWFya2Rvd24ocmVzKSkKCgppZiBfX25hbWVfXyA9PSAnX19tYWluX18nOgogICAgbWFpbigpCg=="
},
"config/complemento_entreno_e2.json": {
"sha256": "a22df590ed7a242810bc579bad443ca8be2700f304af2402a3d2ee27a12f66f3",
"b64": "ewogICJzY2hlbWEiOiAxLAogICJleHBlcmltZW50byI6ICJlMl9jb21wbGVtZW50b19lbnRyZW5hbWllbnRvX3YxIiwKICAiZXN0YWRvIjogIkRpc2XDsWFkbyB5IHByb2JhZG8gZW4gQ1BVIGNvbiBwb2NhcyBpbcOhZ2VuZXM7IE5PIGVqZWN1dGFkbyBlbiBHUFUuIE5pbmfDum4gbsO6bWVybyBkZSBlc3RlIGFyY2hpdm8gZXMgdW4gcmVzdWx0YWRvIGRlbCBjb21wbGVtZW50by4iLAogICJleHBvc2ljaW9uX3Rlc3QiOiAiRWwgdGVzdCB5YSBmdWUgb2JzZXJ2YWRvIGVuIGVsIGJlbmNobWFyayBjZXJyYWRvLiBUb2RvIGxvIGRlIGVzdGUgY29tcGxlbWVudG8gZXMgZXhwbG9yYXRvcmlvOiBzZSBlbGlnZSBzb2xvIGNvbiB2YWxpZGFjacOzbiwgbm8gc2UgcmVhYnJlIGVsIHByb3RvY29sbyBjZXJyYWRvIHksIHNpIHNlIG1pcmEgZWwgdGVzdCwgZXMgdW5hIHNvbGEgcGFzYWRhIGZpbmFsIGRlY2xhcmFkYSBleHBsb3JhdG9yaWEuIiwKICAicHJvdG9jb2xvX2Jhc2UiOiAiY29uZmlnL2V4cGVyaW1lbnRvX2UyLmpzb24iLAogICJjb211biI6IHsKICAgICJzZW1pbGxhcyI6IFsxMSwgMjMsIDM3LCA1MywgNzFdLAogICAgImRhdG9zIjogInRyYWluICgxMDE1KSB5IHZhbCAoMjE4LCAxMjIgZ3J1cG9zKSBjb25nZWxhZG9zOyB0ZXN0IG5vIHNlIGNhcmdhIiwKICAgICJjcml0ZXJpb192YWxpZGFjaW9uIjogIkRpY2UgbWVkaW8gcG9yIGdydXBvIGVuIHZhbGlkYWNpw7NuIChtZWRpYSBkZSBpbcOhZ2VuZXMgZGVudHJvIGRlbCBncnVwbyB5IGx1ZWdvIGVudHJlIGdydXBvcyksIGlndWFsIHF1ZSBlbCBiZW5jaG1hcmsiLAogICAgImNhY2hlX2VuX21lbW9yaWEiOiAiTGFzIGltw6FnZW5lcyB5IG3DoXNjYXJhcyB5YSBwcmVwcm9jZXNhZGFzIHNlIGd1YXJkYW4gZW4gUkFNIGxhIHByaW1lcmEgdmV6IHF1ZSBzZSBsZWVuLiBMb3MgdGVuc29yZXMgc29uIGlkw6ludGljb3MgYSBsb3MgZGUgbG9hZF9leGFtcGxlOyBzb2xvIGNhbWJpYSBlbCB0aWVtcG8uIgogIH0sCiAgImFibGFjaW9uIjogewogICAgImNvbnRyb2xfZ3VpYSI6ICJTZWNjacOzbiA0LjY6IHNvYnJlIGVsIG1lam9yIG1vZGVsbywgYWlzbGFyIGVsIGFwb3J0ZSBkZSBhbCBtZW5vcyB0cmVzIGNvbXBvbmVudGVzLCBjb24gZWwgbWlzbW8gbsO6bWVybyBkZSBzZW1pbGxhcyB5IGNvbiBpbmZlcmVuY2lhIHNvYnJlIGNhZGEgZGlmZXJlbmNpYSAoY2hlY2tsaXN0IDYxKS4iLAogICAgIm1vZGVsbyI6ICJ0cmFuc3VuZXRfMmQiLAogICAgImVsZWNjaW9uX2RlbF9tb2RlbG8iOiAiTWVqb3IgRGljZSBkZSB2YWxpZGFjacOzbiBkZWwgYmVuY2htYXJrICgwLDg1MzIgbWVkaW8gZW4gY2luY28gc2VtaWxsYXMpLiBMYSBlbGVjY2nDs24gbm8gdXNhIHRlc3QuIiwKICAgICJyZWZlcmVuY2lhX2FydGljdWxvIjogIkNoZW4gZXQgYWwuICgyMDIxKSwgVHJhbnNVTmV0OiBUcmFuc2Zvcm1lcnMgTWFrZSBTdHJvbmcgRW5jb2RlcnMgZm9yIE1lZGljYWwgSW1hZ2UgU2VnbWVudGF0aW9uLCBhclhpdjoyMTAyLjA0MzA2LiBFbCBhcnTDrWN1bG8gY29tcGFyYSBSNTAtVS1OZXQsIFI1MC1WaVQtQ1VQIHkgVHJhbnNVTmV0IHkgZXN0dWRpYSBlbCBuw7ptZXJvIGRlIGNvbmV4aW9uZXMgZGUgc2FsdG8gKDAsIDEsIDMpOyBlbnRyZW5hIGNvbiAwLDUgZW50cm9ww61hIGNydXphZGEgKyAwLDUgRGljZS4iLAogICAgInZhcmlhbnRlcyI6IHsKICAgICAgInNpbl90cmFuc2Zvcm1lciI6IHsKICAgICAgICAiY2FtYmlvIjogInRyYW5zZm9ybWVyLm51bV9sYXllcnMgPSAxMiAtPiAwIiwKICAgICAgICAicXVlZGEiOiAiUmVzTmV0LTUwIGjDrWJyaWRhLCBwcm95ZWNjacOzbiBkZSBwYXJjaGVzLCBlbWJlZGRpbmcgcG9zaWNpb25hbCwgTGF5ZXJOb3JtIGZpbmFsLCBkZWNvZGlmaWNhZG9yIENVUCB5IGxhcyB0cmVzIGNvbmV4aW9uZXMgZGUgc2FsdG8iLAogICAgICAgICJwcmVndW50YSI6ICJDdcOhbnRvIGFwb3J0YSBlbCBjb2RpZmljYWRvciBUcmFuc2Zvcm1lciAoMTIgYmxvcXVlcyBkZSBhdXRvYXRlbmNpw7NuKSBzb2JyZSB1biBjb2RpZmljYWRvci1kZWNvZGlmaWNhZG9yIENOTiBlcXVpdmFsZW50ZTsgZXMgZWwgY29udHJhc3RlIFI1MC1VLU5ldCBmcmVudGUgYSBUcmFuc1VOZXQgZGVsIGFydMOtY3VsbyIsCiAgICAgICAgInBlcmRpZGEiOiAiYmNlX2RpY2UiCiAgICAgIH0sCiAgICAgICJzaW5fc2FsdG9zIjogewogICAgICAgICJjYW1iaW8iOiAibl9za2lwID0gMyAtPiAwIiwKICAgICAgICAicXVlZGEiOiAiQ29kaWZpY2Fkb3IgY29tcGxldG8gKFJlc05ldC01MCArIDEyIGJsb3F1ZXMgVHJhbnNmb3JtZXIpIHkgZGVjb2RpZmljYWRvciBlbiBjYXNjYWRhIHNpbiBjb25jYXRlbmFyIG1hcGFzIGRlbCBjb2RpZmljYWRvciIsCiAgICAgICAgInByZWd1bnRhIjogIkN1w6FudG8gYXBvcnRhbiBsYXMgY29uZXhpb25lcyBkZSBzYWx0bzsgZXMgZWwgY2FzbyBuX3NraXAgPSAwIChSNTAtVmlULUNVUCkgZGVsIGVzdHVkaW8gZGUgc2FsdG9zIGRlbCBhcnTDrWN1bG8iLAogICAgICAgICJwZXJkaWRhIjogImJjZV9kaWNlIgogICAgICB9LAogICAgICAic2luX2RpY2UiOiB7CiAgICAgICAgImNhbWJpbyI6ICJww6lyZGlkYSAwLDUgQkNFICsgMCw1IERpY2UgLT4gQkNFIHNvbGEgKHBlc28gMSksIHBvciBpbWFnZW4geSBzb2xvIGVuIGxhIHpvbmEgdsOhbGlkYSIsCiAgICAgICAgInF1ZWRhIjogIkFycXVpdGVjdHVyYSBpZMOpbnRpY2EgYSBsYSBkZWwgYmVuY2htYXJrIiwKICAgICAgICAicHJlZ3VudGEiOiAiQ3XDoW50byBhcG9ydGEgZWwgdMOpcm1pbm8gRGljZSBkZSBsYSBww6lyZGlkYSBjb21wdWVzdGE7IGVsIGFydMOtY3VsbyB1c2EgbGEgbWV6Y2xhLiBBbCBwYXNhciBhIEJDRSBzb2xhIHRhbWJpw6luIGNhbWJpYSBsYSBlc2NhbGEgZGVsIGdyYWRpZW50ZTogc2UgZGVjbGFyYSB5IG5vIHNlIHJlYWp1c3RhIGxhIHRhc2EsIHBvcnF1ZSBsYSByZWNldGEgZGVsIGJlbmNobWFyayBzZSBtYW50aWVuZSBmaWphIiwKICAgICAgICAicGVyZGlkYSI6ICJiY2UiCiAgICAgIH0KICAgIH0sCiAgICAiY29udHJvbF9vcGNpb25hbCI6IHsKICAgICAgImNvbXBsZXRvIjogIlRyYW5zVU5ldCBzaW4gY2FtYmlvcyBjb24gbGEgbWlzbWEgcmVjZXRhLiBTaXJ2ZSBkZSByZWZlcmVuY2lhIGVuIGVsIG1pc21vIGVudG9ybm8gc2kgbGEgR1BVIG8gbGFzIHZlcnNpb25lcyBkaWZpZXJlbiBkZSBsYXMgZGVsIGJlbmNobWFyazsgc2kgbm8gc2UgY29ycmUsIGxhIHJlZmVyZW5jaWEgc29uIGxhcyBjaW5jbyBjb3JyaWRhcyBvcmlnaW5hbGVzLiIKICAgIH0sCiAgICAicmVjZXRhIjogIkxhIGRlIHRyYW5zdW5ldF8yZCBlbiBleHBlcmltZW50b19lMi5qc29uOiBTR0QsIGxyIDAsMDAxLCBtb21lbnRvIDAsOSwgd2VpZ2h0IGRlY2F5IDAsMDAwMSwgbG90ZSAyLCAzMCDDqXBvY2FzLCBwYWNpZW5jaWEgNSwgMjU2IHB4LCBzaW4gcGVzb3MgcHJlZW50cmVuYWRvcyIsCiAgICAiaW5mZXJlbmNpYSI6ICJQb3IgY2FkYSB2YXJpYW50ZSBmcmVudGUgYSBsYSByZWZlcmVuY2lhOiBkaWZlcmVuY2lhIG1lZGlhIHBhcmVhZGEgcG9yIHNlbWlsbGEgY29uIElDOTUgdCBkZSBTdHVkZW50IHkgcHJ1ZWJhIHQgcGFyZWFkYTsgV2lsY294b24gZXhhY3RvIChjb24gY2luY28gcGFyZXMgc3UgcCBtw61uaW1vIGJpbGF0ZXJhbCBlcyAwLDA2MjUpOyBib290c3RyYXAgcGFyZWFkbyBwb3IgZ3J1cG8gZGUgdmFsaWRhY2nDs24gKDIwMDAgcmVtdWVzdHJlb3MpIGUgaW52ZXJzacOzbiBkZSBzaWdub3MgcG9yIGdydXBvICgxMDAwMCk7IEhvbG0gc29icmUgbGFzIHRyZXMgdmFyaWFudGVzLiIsCiAgICAiY29uc2VydmFyIjogImJlc3QiCiAgfSwKICAic29icmVhanVzdGUiOiB7CiAgICAiY29udHJvbF9ndWlhIjogIlNlY2Npw7NuIDExLjE6IHZlcmlmaWNhciBsYSBpbXBsZW1lbnRhY2nDs24gY29uIHVuYSBwcnVlYmEgZGUgc2FuaWRhZCwgc29icmVhanVzdGFyIHVuIGxvdGUgcGVxdWXDsW8gKGNoZWNrbGlzdCA0MSkuIiwKICAgICJtb2RlbG9zIjogWyJmY24iLCAic2VnbmV0IiwgInVuZXQiLCAiZGVlcGxhYnYzcGx1cyIsICJ1bmV0cGx1c3BsdXMiLCAicGlkbmV0X3MiLCAic2VnbmV4dF90IiwgInNlZ2Zvcm1lcl9iMCIsICJ0cmFuc3VuZXRfMmQiLCAidm11bmV0Il0sCiAgICAiaW1hZ2VuZXMiOiA0LAogICAgInNlbWlsbGFfZWxlY2Npb25faW1hZ2VuZXMiOiAyMDI2MTEyMCwKICAgICJzZW1pbGxhX2VudHJlbmFtaWVudG8iOiAxMSwKICAgICJwYXNvcyI6IDUwMCwKICAgICJyZWdpc3RyYXJfY2FkYSI6IDI1LAogICAgInVtYnJhbF9kaWNlIjogMC45LAogICAgInJlZ2xhIjogIlVuIGxvdGUgZmlqbyBkZSA0IGltw6FnZW5lcyBkZSB0cmFpbiwgcHJlc2VudGFkbyBlbnRlcm8gZW4gY2FkYSBwYXNvLCBjb24gZWwgb3B0aW1pemFkb3IgeSBsYSBww6lyZGlkYSBkZSBsYSByZWNldGEgZGUgY2FkYSByZWQuIFBhc2Egc2kgZWwgRGljZSBtZWRpbyBkZSBlc2FzIDQgaW3DoWdlbmVzIGVuIG1vZG8gZXZhbHVhY2nDs24gc3VwZXJhIDAsOTAgYWwgZmluYWwuIEVsIHVtYnJhbCBzZSBmaWphIGFudGVzIGRlIGNvcnJlcjsgc2kgdW5hIHJlZCBubyBsbyBhbGNhbnphIHNlIHJlcG9ydGEgdGFsIGN1YWwsIHNpbiBjYW1iaWFyIGxhIHJlY2V0YS4iCiAgfSwKICAicGVybXV0YWNpb24iOiB7CiAgICAiY29udHJvbF9ndWlhIjogIlRhYmxhIEEgKHNlY2Npw7NuIDcuMyk6IHJlcGV0aXIgZWwgcGlwZWxpbmUgY29tcGxldG8gQiA+PSAxMDAwIHZlY2VzIHBlcm11dGFuZG8geSBlbiB0cmFpbiwgbyBCID49IDIwMCBzaSBlbCBjb3N0byBlcyBhbHRvOyBwID0gKDEgKyAje3NfYiA+PSBzX29ic30pIC8gKEIgKyAxKS4gQ2hlY2tsaXN0IDQxIHkgNTYuIiwKICAgICJxdWVfc2VfcGVybXV0YSI6ICJMYXMgbcOhc2NhcmFzIGRlIHRyYWluIHNlIGJhcmFqYW4gZW50cmUgaW3DoWdlbmVzICh1bmEgcGVybXV0YWNpw7NuIGFsZWF0b3JpYSBkZSBsYXMgMTAxNSBwb3IgcsOpcGxpY2EpLiBDYWRhIGltYWdlbiBjb25zZXJ2YSBzdSB6b25hIHbDoWxpZGE7IGxvcyBww614ZWxlcyBkZSBsYSBtw6FzY2FyYSBhamVuYSBxdWUgY2FlbiBlbiBlbCByZWxsZW5vIG5vIGN1ZW50YW4gZW4gbGEgcMOpcmRpZGEuIFZhbGlkYWNpw7NuIHNlIGV2YWzDumEgY29uIHN1cyBtw6FzY2FyYXMgdmVyZGFkZXJhcy4iLAogICAgInBpcGVsaW5lIjogIklkw6ludGljbyBhbCBiZW5jaG1hcms6IG1pc21vIG9yZGVuIGRlIGxvdGVzIHF1ZSBsYSBzZW1pbGxhIDExLCAzMCDDqXBvY2FzLCBwYWNpZW5jaWEgNSwgbWVqb3Igw6lwb2NhIHBvciBEaWNlIGRlIHZhbGlkYWNpw7NuLiBzX2IgZXMgZWwgbWVqb3IgRGljZSBkZSB2YWxpZGFjacOzbiBkZSBsYSByw6lwbGljYSBiLiIsCiAgICAic2VtaWxsYV9lbnRyZW5hbWllbnRvIjogMTEsCiAgICAic2VtaWxsYV9wZXJtdXRhY2lvbmVzIjogMjAyNjExMjEsCiAgICAic19vYnMiOiAiRGljZSBkZSB2YWxpZGFjacOzbiBkZSBsYSBjb3JyaWRhIG9yaWdpbmFsIGNvbiBsYSBzZW1pbGxhIDExIGRlbCBtaXNtbyBtb2RlbG8gKGJlbmNobWFyayBjZXJyYWRvKSIsCiAgICAiYmxvcXVlcyI6IHsKICAgICAgInBpcGVsaW5lX2ZjbiI6IHsKICAgICAgICAibW9kZWxvIjogImZjbiIsCiAgICAgICAgIkIiOiAyMDAsCiAgICAgICAgIm1vdGl2byI6ICJDb250cm9sIGRlbCBwaXBlbGluZSAoZGF0b3MsIHBhcnRpY2nDs24sIHDDqXJkaWRhLCBldmFsdWFjacOzbiksIHF1ZSBlcyBjb23Dum4gYSBsYXMgMTAgcmVkZXMuIFNlIHVzYSBsYSByZWQgbcOhcyBiYXJhdGEgZGVsIGJlbmNobWFyayBwYXJhIGxsZWdhciBhbCBtw61uaW1vIGRlIGxhIGd1w61hIHBhcmEgbW9kZWxvcyBjb3N0b3NvcyAoQiA9IDIwMDsgcCBtw61uaW1vIDEvMjAxKS4iCiAgICAgIH0sCiAgICAgICJ0cmFuc3VuZXQiOiB7CiAgICAgICAgIm1vZGVsbyI6ICJ0cmFuc3VuZXRfMmQiLAogICAgICAgICJCIjogMTksCiAgICAgICAgIm1vdGl2byI6ICJDb250cm9sIGVzcGVjw61maWNvIGRlbCBtZWpvciBtb2RlbG8uIEIgPSAyMDAgZGUgVHJhbnNVTmV0IG5vIGNhYmUgKGVzdGltYWRvIGRlY2VuYXMgZGUgaG9yYXMgZGUgTDQpLiBDb24gQiA9IDE5IGVsIHAgbcOtbmltbyBlcyAxLzIwID0gMCwwNTogYWxjYW56YSBwYXJhIHJlY2hhemFyIGFsIDUgJSBzb2xvIHNpIG5pbmd1bmEgcsOpcGxpY2EgaWd1YWxhIGEgc19vYnMuIFF1ZWRhIHBvciBkZWJham8gZGVsIG3DrW5pbW8gZGUgbGEgZ3XDrWEgeSBzZSBkZWNsYXJhIGFzw60uIgogICAgICB9CiAgICB9LAogICAgInJlZ2xhX2RlX0IiOiAiQiBzZSBmaWphIGRlc3B1w6lzIGRlbCBwaWxvdG8gZGUgdGllbXBvcyB5IGFudGVzIGRlIHZlciBjdWFscXVpZXIgc19iLiBTaSB1bmEgc2VzacOzbiBzZSBjb3J0YSwgc2UgcmVwb3J0YSBlbCBCIGFsY2FuemFkbyB5IHN1IHAgbcOtbmltbzsgbm8gc2UgZGV0aWVuZSBwb3IgZWwgdmFsb3IgZGUgbG9zIHNfYi4iLAogICAgImNvbnNlcnZhciI6ICJuaW5ndW5vIgogIH0sCiAgImhwbyI6IHsKICAgICJjb250cm9sX2d1aWEiOiAiU2VjY2lvbmVzIDQuMSwgMTEuMSB5IDExLjI6IGLDunNxdWVkYSBhbGVhdG9yaWEgY29uIGVsIG1pc21vIG7Dum1lcm8gZGUgaW50ZW50b3MgVCB5IGVsIG1pc21vIHByZXN1cHVlc3RvIHBvciBtb2RlbG8sIGVzcGFjaW9zIGNvbXBhcmFibGVzIGVuIGFtcGxpdHVkIHJlbGF0aXZhLCBzZWxlY2Npw7NuIHNvbG8gY29uIHZhbGlkYWNpw7NuLCByZWdpc3RybyBkZSB0b2RhcyBsYXMgY29uZmlndXJhY2lvbmVzIChjaGVja2xpc3QgMzcpLiIsCiAgICAibWV0b2RvIjogIkLDunNxdWVkYSBhbGVhdG9yaWEgY29uIHVuYSBzZWN1ZW5jaWEgY29tw7puIGRlIG11bHRpcGxpY2Fkb3JlczogZWwgaW50ZW50byB0IHVzYSBlbCBtaXNtbyBtdWx0aXBsaWNhZG9yIGRlIHRhc2EgeSBlbCBtaXNtbyB3ZWlnaHQgZGVjYXkgZW4gbGFzIDEwIHJlZGVzLCBhcGxpY2FkbyBzb2JyZSBsYSByZWNldGEgZGUgY2FkYSB1bmEuIEVsIGludGVudG8gMCBlcyBsYSByZWNldGEgZGVsIGJlbmNobWFyay4iLAogICAgImVzcGFjaW8iOiB7CiAgICAgICJsciI6ICJyZWNldGEgeCAxMF51LCB1IH4gVW5pZm9ybWUoLTEsIDEpIChlc2NhbGEgbG9nYXLDrXRtaWNhLCB1bmEgZMOpY2FkYSBoYWNpYSBjYWRhIGxhZG8pIiwKICAgICAgIndlaWdodF9kZWNheSI6ICJ1bm8gZGUgezAsIDAsMDAwMSwgMCwwMX0gY29uIGlndWFsIHByb2JhYmlsaWRhZCIsCiAgICAgICJmaWpvcyI6ICJvcHRpbWl6YWRvciBkZSBsYSByZWNldGEsIGxvdGUgMiwgMjU2IHB4LCBhcnF1aXRlY3R1cmEsIHDDqXJkaWRhIgogICAgfSwKICAgICJzZW1pbGxhX3NlY3VlbmNpYSI6IDIwMjYxMTIyLAogICAgIlRfZGlzZW5hZG8iOiA4LAogICAgIlRfcmVjb21lbmRhZG9fYW50ZXNfZGVsX2RvbWluZ28iOiA0LAogICAgInByZXN1cHVlc3RvX3Bvcl9pbnRlbnRvIjogeyJlcG9jYXMiOiAxMCwgInBhY2llbmNpYSI6IDMsICJzZW1pbGxhIjogMTF9LAogICAgIm9yZGVuIjogIlBvciBpbnRlbnRvIHkgbHVlZ28gcG9yIHJlZDogc2kgbGEgc2VzacOzbiBzZSBjb3J0YSwgdG9kbyBwcmVmaWpvIGNvbXBsZXRvIGRlIGludGVudG9zIHRpZW5lIGVsIG1pc21vIHByZXN1cHVlc3RvIHBhcmEgbGFzIDEwLiIsCiAgICAiZmFzZV8yX2Rpc2VuYWRhX25vX3Byb2dyYW1hZGEiOiAiUmVlbnRyZW5hciBsYSBjb25maWd1cmFjacOzbiBnYW5hZG9yYSBkZSBjYWRhIHJlZCBjb24gbGFzIGNpbmNvIHNlbWlsbGFzIHkgZWwgcHJvdG9jb2xvIGNvbXBsZXRvICgzMCDDqXBvY2FzLCBwYWNpZW5jaWEgNSkuIFVuYXMgMTcgaG9yYXMgZGUgTDQgZXN0aW1hZGFzOyBubyBjYWJlIGFudGVzIGRlbCBkb21pbmdvLiIsCiAgICAibGltaXRhY2lvbmVzIjogIlVuYSBzZW1pbGxhIHBvciBpbnRlbnRvIHkgMTAgw6lwb2NhczogbGEgZWxlY2Npw7NuIHRpZW5lIHJ1aWRvIGRlIHNlbWlsbGEgeSBmYXZvcmVjZSBjb25maWd1cmFjaW9uZXMgcXVlIGFwcmVuZGVuIHLDoXBpZG8uIEVsIGVzcGFjaW8gc2UgbGltaXRhIGEgdGFzYSB5IHdlaWdodCBkZWNheSwgbG9zIGRvcyBoaXBlcnBhcsOhbWV0cm9zIGNvbXVuZXMgYSBsb3MgMTAgb3B0aW1pemFkb3Jlczsgbm8gY3VicmUgbG90ZSwgZHJvcG91dCwgcHJvZnVuZGlkYWQgbmkgYW5jaG8gcXVlIHBpZGUgbGEgZ3XDrWEgY29tbyBlc3BhY2lvIG3DrW5pbW8uIiwKICAgICJjb25zZXJ2YXIiOiAibmluZ3VubyIKICB9LAogICJyZWZlcmVuY2lhX29yaWdpbmFsIjogewogICAgImZ1ZW50ZSI6ICJyZXN1bHRhZG9zL2F1ZGl0b3JpYV9lMl9yZWNpYmlkYS9hdWRpdG9yaWEuanNvbiAodmFsaWRhY2nDs24sIG1lam9yIMOpcG9jYSwgNTAgY29ycmlkYXMgZGVsIGJlbmNobWFyaykiLAogICAgImRpY2VfdmFsaWRhY2lvbl9wb3Jfc2VtaWxsYSI6IHsKICAgICAgInRyYW5zdW5ldF8yZCI6IHsiMTEiOiAwLjg1NTM5MzQ3NTY1MDQ4NSwgIjIzIjogMC44NTY0MTQ5NDA5MjQzMjg4LCAiMzciOiAwLjg2MTAxMTY2NzIyOTk1NzQsICI1MyI6IDAuODQ3Njk1NzY0MTE0NTcxNiwgIjcxIjogMC44NDU2OTEyOTU2NDYzNjY4fSwKICAgICAgImZjbiI6IHsiMTEiOiAwLjgyMzA0MjUyNTY3MTk2NzIsICIyMyI6IDAuODMyMDIwODg3MjM5NDcxLCAiMzciOiAwLjgzMDI2MjY0MjQ0ODgzNTcsICI1MyI6IDAuODI3MjQ0NTk0OTE2MjMxMywgIjcxIjogMC44NDIwNDg5OTgyMDYwNjczfQogICAgfSwKICAgICJzZWd1bmRvc19wb3JfZXBvY2FfbDQiOiB7ImZjbiI6IDE5LjYsICJzZWduZXQiOiAxOS42LCAidW5ldCI6IDE5LjcsICJkZWVwbGFidjNwbHVzIjogMzQuOCwgInVuZXRwbHVzcGx1cyI6IDM2LjEsICJwaWRuZXRfcyI6IDcwLjUsICJzZWduZXh0X3QiOiA2NS4xLCAic2VnZm9ybWVyX2IwIjogNDguMCwgInRyYW5zdW5ldF8yZCI6IDEwMi4xLCAidm11bmV0IjogMTA2Ljd9LAogICAgImVwb2Nhc19tZWRpYXMiOiB7ImZjbiI6IDI5LjIsICJzZWduZXQiOiAxOS42LCAidW5ldCI6IDI3LjAsICJkZWVwbGFidjNwbHVzIjogMjEuMiwgInVuZXRwbHVzcGx1cyI6IDIwLjAsICJwaWRuZXRfcyI6IDIzLjQsICJzZWduZXh0X3QiOiAyMi42LCAic2VnZm9ybWVyX2IwIjogMjQuMCwgInRyYW5zdW5ldF8yZCI6IDI0LjQsICJ2bXVuZXQiOiAxOS42fSwKICAgICJub3RhX3RpZW1wb3MiOiAiU2VndW5kb3MgdG90YWxlcyBkZSBsYSBjb3JyaWRhIGRpdmlkaWRvcyBwb3Igc3VzIMOpcG9jYXM7IGluY2x1eWVuIGxhIHZhbGlkYWNpw7NuIGRlIGNhZGEgw6lwb2NhIHkgbGEgbGVjdHVyYSBkZSBpbcOhZ2VuZXMgZGVzZGUgZGlzY28uIgogIH0KfQo="
}
}
for rel, item in INCRUSTADOS.items():
    data = base64.b64decode(item['b64'])
    assert hashlib.sha256(data).hexdigest() == item['sha256'], rel
    destino = PROJECT/rel
    destino.write_bytes(data)
    print('escrito', rel, item['sha256'][:12])

## Entorno aislado, igual al del benchmark
Python 3.11 fuera del kernel, torch 2.5.1+cu121, `requirements-e2.txt`, transformers 4.46.3 y las
ruedas de causal-conv1d y mamba-ssm, como en `Benchmark_E2_Colab_v2.ipynb`. Si `/content` se borró
(sesión nueva), esta celda lo reconstruye. Al final compara el `pip freeze` con el del benchmark.

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','uv==0.6.9'],check=True)
ENV=Path('/content/imo-e2-env')
if not (ENV/'bin/python').exists():
    subprocess.run([sys.executable,'-m','uv','venv','--python','3.11',str(ENV)],check=True)
PY=str(ENV/'bin/python')
def pip_install(*items):
    subprocess.run([sys.executable,'-m','uv','pip','install','--python',PY,*items],check=True)
pip_install('setuptools==75.8.0')
pip_install('torch==2.5.1','torchvision==0.20.1','--index-url','https://download.pytorch.org/whl/cu121')
pip_install('-r',str(PROJECT/'requirements-e2.txt'))
pip_install('transformers==4.46.3')
pip_install('--no-deps','https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/causal_conv1d-1.5.0.post8%2Bcu12torch2.5cxx11abiFALSE-cp311-cp311-linux_x86_64.whl')
pip_install('--no-deps','https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4%2Bcu12torch2.5cxx11abiFALSE-cp311-cp311-linux_x86_64.whl')
# Como en Complemento_GPU_E2: las extensiones de Mamba cargan libc10/libcudart del venv, no las del sistema.
site=ENV/'lib/python3.11/site-packages'
dirs=[site/'torch/lib',*sorted((site/'nvidia').glob('*/lib'))]
paths=[str(p) for p in dirs if p.is_dir()]+os.environ.get('LD_LIBRARY_PATH','').split(':')
os.environ['LD_LIBRARY_PATH']=':'.join(dict.fromkeys(p for p in paths if p))
subprocess.run([PY,'-c','import torch, torchvision, mamba_ssm; assert torch.cuda.is_available(); print(torch.__version__,torch.cuda.get_device_name(0))'],check=True)
freeze=subprocess.check_output([sys.executable,'-m','uv','pip','freeze','--python',PY],text=True)
(OUTPUT/'environment.freeze.txt').write_text(freeze)
original = BASE_RUNS/'environment.freeze.txt'
if original.exists():
    a, b = set(original.read_text().splitlines()), set(freeze.splitlines())
    print('Paquetes distintos del benchmark:', sorted(a ^ b) or 'ninguno')
else:
    print('No se encontró el freeze del benchmark en', original)

## Comprobación previa (la misma del benchmark) y lanzador con registro
`preflight_e2.py` prueba forward, gradientes y un paso de optimizador de las 10 redes con datos
sintéticos y deja `preflight.json` en la carpeta del complemento. El lanzador escribe cada salida
en `logs/` dentro de Drive además de mostrarla aquí.

In [ ]:
MODELS=['fcn','segnet','unet','deeplabv3plus','unetplusplus','pidnet_s','segnext_t','segformer_b0','transunet_2d','vmunet']
subprocess.run([PY,str(PROJECT/'scripts/preflight_e2.py'),'--output',str(OUTPUT),'--models',*MODELS],check=True)
report=json.loads((OUTPUT/'preflight.json').read_text())
for name,result in report['models'].items(): print(name,result['status'])
assert all(r['status']=='passed' for r in report['models'].values()), 'No lanzar bloques con un preflight fallido'

RUNNER = str(PROJECT/'scripts/complemento_entreno_e2.py')
def correr(*argumentos):
    sello = datetime.datetime.now().strftime('%Y%m%dT%H%M%S')
    log = OUTPUT/'logs'/f'{argumentos[0]}_{sello}.log'
    comando = ' '.join([PY, '-u', RUNNER, *map(str, argumentos), '--output', str(OUTPUT)])
    print('$', comando, '\n  log:', log)
    subprocess.run(['bash', '-o', 'pipefail', '-c', f'cd {PROJECT}/scripts && {comando} 2>&1 | tee -a {log}'], check=True)

## Piloto: medir antes de lanzar
Corre la primera época de tres corridas reales de la ablación (sin Dice, que cuesta lo mismo que
TransUNet completo; sin Transformer; sin saltos) y dos épocas de la primera permutación de FCN.
Esas épocas **no se pierden**: cada bloque reanuda desde ellas. Luego escribe `estimacion_computo.json`
con horas de L4 por bloque, marcadas como estimadas. Con esa tabla se fijan `B` y `T` **antes**
de ver ningún resultado de permutación o de HPO. Tarda unos 6 minutos estimados.

In [ ]:
correr('piloto')
print(json.dumps(json.loads((OUTPUT/'estimacion_computo.json').read_text()), indent=2, ensure_ascii=False))

## Bloque 1 — sobreajuste a un lote (10 redes, unos minutos estimados)
Lote fijo de 4 imágenes de train, 500 pasos con la receta de cada red. Pasa si el Dice de esas 4 en
modo evaluación supera 0,90. Si una red no pasa, se reporta tal cual: no se cambia su receta.

In [ ]:
EJECUTAR_SOBREAJUSTE = False
if EJECUTAR_SOBREAJUSTE:
    correr('sobreajuste')

## Bloque 2 — ablación de TransUNet (15 corridas, unas 9 horas de L4 estimadas)
Tres variantes × semillas 11, 23, 37, 53, 71, con la receta del benchmark. `CONTROL=True` añade
TransUNet completo (5 corridas más, unas 3,5 h) como referencia en este mismo entorno; sin él, la
referencia son las corridas originales. Para repartir entre dos sesiones de Colab a la vez, usa
`FRAGMENTO='0/2'` en una y `'1/2'` en la otra (misma carpeta de Drive, tareas disjuntas).

In [ ]:
EJECUTAR_ABLACION = False
CONTROL = False
FRAGMENTO = None   # p. ej. '0/2'
if EJECUTAR_ABLACION:
    variantes = ['sin_transformer','sin_saltos','sin_dice'] + (['completo'] if CONTROL else [])
    correr('ablacion', '--variantes', *variantes, *(['--fragmento', FRAGMENTO] if FRAGMENTO else []))

## Bloque 3 — permutación del pipeline con FCN (B = 200)
Mínimo de la guía para modelos costosos. Es el control del pipeline común a las 10 redes. Fija `B`
con la estimación del piloto y no lo cambies después de ver resultados. Se puede repartir con `FRAGMENTO`.

In [ ]:
EJECUTAR_PERMUTACION_FCN = False
B_FCN = 200
FRAGMENTO = None
if EJECUTAR_PERMUTACION_FCN:
    correr('permutacion', '--permutacion', 'pipeline_fcn', '--B', B_FCN, *(['--fragmento', FRAGMENTO] if FRAGMENTO else []))

## Bloque 4 — búsqueda de hiperparámetros con igual presupuesto (T intentos × 10 redes)
Intento 0 = receta; intentos 1..T-1 = multiplicador de tasa 10^u y weight decay comunes a las 10.
Presupuesto por intento: semilla 11, hasta 10 épocas, paciencia 3. Las tareas van por intento y luego
por red, así que cualquier corte deja un prefijo completo igual para todas. Recomendado `T=4`.

In [ ]:
EJECUTAR_HPO = False
T = 4
FRAGMENTO = None
if EJECUTAR_HPO:
    correr('hpo', '--T', T, *(['--fragmento', FRAGMENTO] if FRAGMENTO else []))

## Bloque 5 — permutación específica de TransUNet (B = 19)
Con B = 19 el p mínimo es 0,05: por debajo del mínimo de la guía (B ≥ 200), se declara así.

In [ ]:
EJECUTAR_PERMUTACION_TRANSUNET = False
B_TRANSUNET = 19
FRAGMENTO = None
if EJECUTAR_PERMUTACION_TRANSUNET:
    correr('permutacion', '--permutacion', 'transunet', '--B', B_TRANSUNET, *(['--fragmento', FRAGMENTO] if FRAGMENTO else []))

## Resumen
Lee solo los JSON de la carpeta del complemento y, para la referencia de la ablación y el s_obs de la
permutación, los `result.json` y mejores checkpoints de validación del benchmark original. Escribe
`RESUMEN.json` y `RESUMEN.md` en Drive. Descarga esos dos archivos, `estimacion_computo.json`,
`environment.freeze.txt`, `preflight.json`, la carpeta `logs/` y los `result.json`/`history.json` de
cada corrida; los checkpoints de la ablación se quedan en Drive.

In [ ]:
subprocess.run([PY, str(PROJECT/'scripts/resumir_complemento_entreno_e2.py'), '--output', str(OUTPUT),
                '--base-runs', str(BASE_RUNS)], check=True)
abiertas = sorted(p.parent.name for p in OUTPUT.glob('*/last.pt'))
print('Corridas empezadas sin cerrar:', abiertas or 'ninguna')